# Synthetic Ramesside Star Clocks

In [1]:
import numpy as np
import sys 
from pathlib import Path
import pandas as pd

# Set up paths
PROJECT_ROOT = Path().resolve().parents[0]
sys.path.append(str(PROJECT_ROOT))
#import decanopy

# I want get_sun to shut up so ignoring warnings:
import warnings
warnings.simplefilter('ignore', UserWarning)


## Import Sky & Initialize 

Due to ongoing code cleaning, the pathing for the inputs and outputs is still being reworked a little. For now, specify one of two runs manually: 
- Option 1: "real sky", aka Hipparcos data for 1300 BCE. 
- Option 2: "rand sky", aka one specfic procedurally generated sky with 518 stars

The data is organized as follows. In each case,`[type]` = `real_sky` or `rand_sky`. 
- The night sky data input files (star names, Dec/RAs, & mags) are stored in `/decanOpy/data/input/skyflow/[type]`.
- The night sky data output ("skyflow") outputs are stored in `/decanOpy/data/output/skyflow/[type]`.
- The synthetic RSCs are stored in `/decanOpy/data/output/rsc/[type]`. 

### Option 1: Run this for real_sky

In [2]:
from decanopy.config import DEFAULT_STAR_DATA, SKYFLOW_OUTPUT_REAL_SKY, RSC_OUTPUT_REAL_SKY, DEFAULT_STAR_NAMES
filename = "hippdata1300BC.txt"
filepath = SKYFLOW_OUTPUT_REAL_SKY / filename 

# make dictionary of names and magnitudes (if not already in memory) 
name_df = pd.read_csv(DEFAULT_STAR_NAMES, index_col=None, header=0, names=['Name', 'RA', 'Dec', 'Mag'])
mag_dict = name_df.set_index('Name')['Mag'].to_dict()

# determine output writepath
writepath = RSC_OUTPUT_REAL_SKY 

### Option 2: Run this for rand_sky

In [3]:
from decanopy.config import SKYFLOW_OUTPUT_RAND_SKY, USER_INPUT_RAND_SKY, RSC_OUTPUT_RAND_SKY
filename = 'mockdata_518_1300BC-Mar-18-2024_1059.txt'
filepath = SKYFLOW_OUTPUT_RAND_SKY / filename 

# make dictionary of names and magnitudes (if not already in memory) 
ic_filename = 'star_data_Mar-18-2024_1059.csv'
name_df = pd.read_csv(USER_INPUT_RAND_SKY / ic_filename, index_col=None, header=0, names=['Name', 'RA', 'Dec', 'Mag'])
mag_dict = name_df.set_index('Name')['Mag'].to_dict()

# determine output writepath
writepath = RSC_OUTPUT_RAND_SKY 

## Initalize Sky

In [3]:
from decanopy.models.RSC.syn_rsc import initialize_sky

skydict = initialize_sky(filepath)

## Create Synthetic RSCs

In [4]:
# Define parameters for synRSC model 
alt_window = (0, 30) # degrees 
horizon = (165, 195) # Note: MUST have smaller number first; 
bsize = 1 # bin size (must be 1 if gsize = 0)
gsize = 0 # gap size (relative to binsize) 

# Name output file
num_decs = len(skydict['starlist']) #used only for naming convention
writename = filename[0:-4] + str(horizon[0]) + '-' + str(horizon[1]) + '_b=' + str(bsize) + '_g=' + str(gsize) + '_' + str(num_decs) + '.xlsx'
writename = 'NEWNEWtestname' + '.xlsx' ## uncomment and edit for user-specified filename

from decanopy.models.RSC.syn_rsc import write_synRSC_to_excel

# write excel file
write_synRSC_to_excel(
    writepath, writename, horizon, alt_window, bsize, gsize, skydict, mag_dict
)